In [2]:
# core libraries
import numpy as np
import pandas as pd
import re
#visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Ml 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

#Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Open this file "TwitterHate.csv"

In [4]:
df = pd.read_csv("TwitterHate.csv")


In [5]:
df.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


In [6]:
df.tail(10)

,id,label,tweet
31952,31953,0,@user you went too far with @user
31953,31954,0,good morning #instagram #shower #water #berlin...
31954,31955,0,#holiday bull up: you will dominate your bul...
31955,31956,0,less than 2 weeks ð ðð¼ð¹ððµ @us...
31956,31957,0,off fishing tomorrow @user carnt wait first ti...
31957,31958,0,ate @user isz that youuu?ðððððð...
31958,31959,0,to see nina turner on the airwaves trying to...
31959,31960,0,listening to sad songs on a monday morning otw...
31960,31961,1,"@user #sikh #temple vandalised in in #calgary,..."
31961,31962,0,thank you @user for you follow


In [7]:
df['label'].unique()

array([0, 1], dtype=int64)

In [8]:
df.shape

(31962, 3)

In [9]:
#Convert tweets column into a list

In [10]:
tweets = df['tweet'].astype(str).tolist()

In [11]:
labels = df['label'].tolist()

In [12]:
tweets[:5]

[' @user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction.   #run',
 "@user @user thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx.    #disapointed #getthanked",
 '  bihday your majesty',
 '#model   i love u take with u all the time in urð\x9f\x93±!!! ð\x9f\x98\x99ð\x9f\x98\x8eð\x9f\x91\x84ð\x9f\x91\x85ð\x9f\x92¦ð\x9f\x92¦ð\x9f\x92¦  ',
 ' factsguide: society now    #motivation']

In [13]:
labels[:5]

[0, 0, 0, 0, 0]

In [14]:
len(tweets)

31962

In [15]:
# Tweet Cleanup

In [16]:
#Normalize the casing (lowercase)
tweets = [tweet.lower() for tweet in tweets]

In [34]:
# Remove user handles (@username)

In [36]:
tweets = [re.sub(r'@\w+', '', tweet) for tweet in tweets]

In [38]:
tweets[:5]

['  when a father is dysfunctional and is so selfish he drags his kids into his dysfunction.   #run',
 "  thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx.    #disapointed #getthanked",
 '  bihday your majesty',
 '#model   i love u take with u all the time in urð\x9f\x93±!!! ð\x9f\x98\x99ð\x9f\x98\x8eð\x9f\x91\x84ð\x9f\x91\x85ð\x9f\x92¦ð\x9f\x92¦ð\x9f\x92¦  ',
 ' factsguide: society now    #motivation']

In [40]:
# Remove URLs

In [42]:
tweets = [re.sub(r'http\S+|www.\S+', '', tweet) for tweet in tweets]

In [51]:
# Tokenize tweets using TweetTokenizer

In [55]:
from nltk.tokenize import TweetTokenizer


In [57]:
tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)

In [59]:
tweets_tokens = [tokenizer.tokenize(tweet) for tweet in tweets]

In [61]:
# Remove stopwords

In [63]:
from nltk.corpus import stopwords

In [65]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [67]:
stop_words = set(stopwords.words('english'))

tweets_tokens = [
    [word for word in tweet if word not in stop_words]
    for tweet in tweets_tokens
]

In [69]:
# Remoe redundant terms: -

In [71]:
redundant_words = ['rt', 'amp']

In [73]:
tweets_tokens = [
    [word for word in tweet if word not in redundant_words]
    for tweet in tweets_tokens
]

In [75]:
# Remove # symbol but keep the word

In [77]:
tweets_tokens = [
    [word.replace('#', '') for word in tweet]
    for tweet in tweets_tokens
]

In [79]:
tweets_tokens[:5]

[['father',
  'dysfunctional',
  'selfish',
  'drags',
  'kids',
  'dysfunction',
  '.',
  'run'],
 ['thanks',
  'lyft',
  'credit',
  "can't",
  'use',
  'cause',
  'offer',
  'wheelchair',
  'vans',
  'pdx',
  '.',
  'disapointed',
  'getthanked'],
 ['bihday', 'majesty'],
 ['model',
  'love',
  'u',
  'take',
  'u',
  'time',
  'urð',
  '\x9f',
  '\x93',
  '±',
  '!',
  '!',
  '!',
  'ð',
  '\x9f',
  '\x98',
  '\x99',
  'ð',
  '\x9f',
  '\x98',
  '\x8e',
  'ð',
  '\x9f',
  '\x91',
  '\x84',
  'ð',
  '\x9f',
  '\x91',
  'ð',
  '\x9f',
  '\x92',
  '¦',
  'ð',
  '\x9f',
  '\x92',
  '¦',
  'ð',
  '\x9f',
  '\x92',
  '¦'],
 ['factsguide', ':', 'society', 'motivation']]

In [81]:
# Remove terms with length = 1

In [83]:
tweets_tokens = [
    [word for word in tweet if len(word) > 1]
    for tweet in tweets_tokens
]

In [85]:
tweets_tokens[:5]

[['father', 'dysfunctional', 'selfish', 'drags', 'kids', 'dysfunction', 'run'],
 ['thanks',
  'lyft',
  'credit',
  "can't",
  'use',
  'cause',
  'offer',
  'wheelchair',
  'vans',
  'pdx',
  'disapointed',
  'getthanked'],
 ['bihday', 'majesty'],
 ['model', 'love', 'take', 'time', 'urð'],
 ['factsguide', 'society', 'motivation']]

In [87]:
# Check Top Terms in Tweets

In [89]:
# Flatten all tokens into one list

In [91]:
all_tokens = []
for tweet in tweets_tokens:
    all_tokens.extend(tweet)

In [93]:
from collections import Counter

In [95]:
# Find 10 most common terms

In [97]:
word_freq = Counter(all_tokens)
word_freq.most_common(10)

[('...', 2807),
 ('love', 2748),
 ('day', 2276),
 ('happy', 1684),
 ('time', 1131),
 ('life', 1118),
 ('like', 1047),
 ('today', 1013),
 ('new', 994),
 ('thankful', 946)]

In [99]:
# Data Formatting for Predictive Modeling

In [101]:
# Join tokens back into strings

In [103]:
tweets_cleaned = [' '.join(tweet) for tweet in tweets_tokens]

In [105]:
tweets_cleaned[:3]

['father dysfunctional selfish drags kids dysfunction run',
 "thanks lyft credit can't use cause offer wheelchair vans pdx disapointed getthanked",
 'bihday majesty']

In [107]:
# Assign features and target

In [109]:
X = tweets_cleaned
y = labels

In [111]:
# Split data into training and testing sets

In [113]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [115]:
len(X_train), len(X_test)

(25569, 6393)

In [117]:
# TF-IDF Vectorization

In [119]:
# Import TF-IDF Vectorizer

In [127]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [129]:
# Instantiate TF-IDF (max 5000 features)

In [131]:
tfidf = TfidfVectorizer(max_features=5000)

In [133]:
# Fit & transform on training data

In [135]:
X_train_tfidf = tfidf.fit_transform(X_train)

In [137]:
# Transform test data

In [139]:
X_test_tfidf = tfidf.transform(X_test)

In [141]:
X_train_tfidf.shape, X_test_tfidf.shape

((25569, 5000), (6393, 5000))

In [145]:
# Model Building (Ordinary Logistic Regression)

In [149]:
from sklearn.linear_model import LogisticRegression

In [151]:
# Instantiate Logistic Regression (default parameters)

In [153]:
lr_model = LogisticRegression()

In [155]:
# Fit model on training data

In [157]:
lr_model.fit(X_train_tfidf, y_train)

LogisticRegression()

In [159]:
# Make predictions (train & test)

In [161]:
# Predictions on training data

In [163]:
y_train_pred = lr_model.predict(X_train_tfidf)

In [165]:
# Predictions on test data

In [167]:
y_test_pred = lr_model.predict(X_test_tfidf)

In [169]:
y_train_pred[:10], y_test_pred[:10]

(array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))

In [171]:
# Model Evaluation (Train Set)

In [173]:
# Accuracy on training set

In [175]:
from sklearn.metrics import accuracy_score, recall_score, f1_score

In [177]:
# Training accuracy

In [179]:
train_accuracy = accuracy_score(y_train, y_train_pred)
train_accuracy

0.9558058586569674

In [181]:
# Training recall

In [183]:
train_recall = recall_score(y_train, y_train_pred)
train_recall

0.39409141583054624

In [191]:
if train_recall > 0.8:
    recall_level = "High"
elif train_recall > 0.6:
    recall_level = "Descent"
else:
    recall_level = "Low"
recall_level

'Low'

In [193]:
# F1 score on training set

In [195]:
train_f1 = f1_score(y_train, y_train_pred)
train_f1

0.5558176100628931

# The Logistic Regression model achieved high training accuracy (~95.6%), indicating strong overall classification performance.
# However, the low recall and moderate F1-score suggest that 
# the model misses a significant portion of hate speech tweets, highlighting the need for further tuning or handling class imbalance.

In [198]:
# Handle Class Imbalance in Logistic Regression

In [200]:
# Adjust class weights in Logistic Regression

In [204]:
lr_balanced = LogisticRegression(class_weight='balanced')

In [206]:
# Fit balanced model on training data

In [208]:
lr_balanced.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced')

In [210]:
# Make predictions again

In [212]:
y_train_pred_bal = lr_balanced.predict(X_train_tfidf)
y_test_pred_bal = lr_balanced.predict(X_test_tfidf)

In [214]:
# Re-evaluate (TRAIN ste): -

In [216]:
# Training accuracy (balanced)

In [218]:
accuracy_score(y_train, y_train_pred_bal)

0.9469279205287653

In [220]:
# Training recall (balanced)

In [224]:
recall_score(y_train, y_train_pred_bal)

0.9654403567447045

In [226]:
# Training F1 score (balanced)

In [228]:
f1_score(y_train, y_train_pred_bal)

0.7185231279817466

# After handling class imbalance using class-weighted Logistic Regression, a significant 
# improvement was observed in recall and F1-score. Recall increased substantially, and the F1-score 
# improved from ~0.55 to ~0.71 while maintaining good
# training accuracy, indicating that the model has learned to identify hate speech more effectively.

In [231]:
# Retrain & Evaluate (After Class Imbalance Adjustment)

In [233]:
# Train the balanced model on training set

In [235]:
lr_balanced.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced')

In [237]:
# Predictions on training set

In [239]:
y_train_pred_bal = lr_balanced.predict(X_train_tfidf)

In [241]:
# Evaluate the model (TRAIN set)

In [243]:
#  Accuracy on training set

In [245]:
train_accuracy_bal = accuracy_score(y_train, y_train_pred_bal)

In [251]:
# Recall on training set

In [253]:
train_recall_bal = recall_score(y_train, y_train_pred_bal)

In [255]:
# Recall on training ste

In [257]:
train_f1_bal = f1_score(y_train, y_train_pred_bal)

In [259]:
train_accuracy_bal, train_recall_bal, train_f1_bal

(0.9469279205287653, 0.9654403567447045, 0.7185231279817466)

In [261]:
# Regularization & Hyperparameter Tuning

In [263]:
# Import GridSearchCV & StratifiedKFold

In [289]:
# Fit GridSearch on training data

In [295]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet', None],
    'C': [0.1, 1, 10],
    'solver': ['liblinear', 'saga']  # solver must match penalty
}

grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5)
grid.fit(X_train_tfidf, y_train)


GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000),
             param_grid={'C': [0.1, 1, 10],
                         'penalty': ['l1', 'l2', 'elasticnet', None],
                         'solver': ['liblinear', 'saga']})

In [308]:
# Best parameters & score check

In [310]:
best_lr = grid.best_estimator_

In [312]:
print(best_lr)

LogisticRegression(C=10, max_iter=1000, solver='saga')


In [314]:
#Best model se TRAIN predictions

In [316]:
y_train_pred_best = best_lr.predict(X_train_tfidf)

In [318]:
# Evaluate TRAIN set (final tuned model)

In [320]:
train_accuracy_best = accuracy_score(y_train, y_train_pred_best)
train_recall_best = recall_score(y_train, y_train_pred_best)
train_f1_best = f1_score(y_train, y_train_pred_best)

train_accuracy_best, train_recall_best, train_f1_best


(0.9812663772537057, 0.7597547380156076, 0.8505460218408737)

In [322]:
# STEP 12: GridSearch with class imbalance handling

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ),
    param_grid,
    cv=5,
    scoring='f1'
)

grid.fit(X_train_tfidf, y_train)


GridSearchCV(cv=5,
             estimator=LogisticRegression(class_weight='balanced',
                                          max_iter=1000),
             param_grid={'C': [0.1, 1, 10],
                         'penalty': ['l1', 'l2', 'elasticnet', None],
                         'solver': ['liblinear', 'saga']},
             scoring='f1')

In [323]:
# Best tuned model
best_lr = grid.best_estimator_

# Train predictions using best model
y_train_pred_best = best_lr.predict(X_train_tfidf)

# Final evaluation on train set
accuracy_score(y_train, y_train_pred_best)
recall_score(y_train, y_train_pred_best)
f1_score(y_train, y_train_pred_best)


0.814187643020595

In [324]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Stratified 4-Fold Cross Validation
skf_recall = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

# GridSearch focused on Recall
grid_recall = GridSearchCV(
    LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ),
    param_grid=param_grid,
    cv=skf_recall,
    scoring='recall'
)

# Fit on training data
grid_recall.fit(X_train_tfidf, y_train)


GridSearchCV(cv=StratifiedKFold(n_splits=4, random_state=42, shuffle=True),
             estimator=LogisticRegression(class_weight='balanced',
                                          max_iter=1000),
             param_grid={'C': [0.1, 1, 10],
                         'penalty': ['l1', 'l2', 'elasticnet', None],
                         'solver': ['liblinear', 'saga']},
             scoring='recall')

In [325]:
grid_recall.best_params_


{'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}

In [326]:
grid_recall.best_score_


0.7854010897231944

In [327]:
best_lr_recall = grid_recall.best_estimator_

y_train_pred_recall = best_lr_recall.predict(X_train_tfidf)

accuracy_score(y_train, y_train_pred_recall)
recall_score(y_train, y_train_pred_recall)
f1_score(y_train, y_train_pred_recall)


0.7189705271897052

In [328]:
grid_recall.best_params_


{'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}

In [329]:
# Best estimator optimized for recall
best_lr_final = grid_recall.best_estimator_


In [330]:
# Predictions on test set
y_test_pred_final = best_lr_final.predict(X_test_tfidf)


In [331]:
# Recall on test set
test_recall = recall_score(y_test, y_test_pred_final)
test_recall


0.7834821428571429

In [332]:
# F1 score on test set
test_f1 = f1_score(y_test, y_test_pred_final)
test_f1


0.585